# Tarea de Cassandra: CRUD Completo y Modelado (Red Social & Blog de Publicaciones Técnicas)

En esta tarea modelaremos el almacenamiento distribuido para las publicaciones técnicas de una red social.
Aprenderemos a diseñar esquemas DDL distribuidos, particiones compuestas con ordenamiento por clave de clustering, operaciones de inserción escalar y compleja (`set`, `list`, `map`), `USING TTL`, consultas DQL de rango e `IN`, mutaciones atómicas en colecciones, eliminaciones granulares y transacciones `BATCH`.
Aprenderemos a diseñar esquemas DDL distribuidos, particiones compuestas con ordenamiento por clave de clustering, operaciones de inserción escalar y compleja (`set`, `list`, `map`), `USING TTL`, consultas DQL de rango e `IN`, mutaciones atómicas en colecciones, eliminaciones granulares y transacciones `BATCH`.

## Sección 1 - Creación de la conexión

Edita los parámetros de conexión en la celda de abajo.

In [4]:
cassandra_user = "cassandra"
cassandra_password = "cassandra"
cassandra_endpoints = ["127.0.0.1:9041", "127.0.0.1:9042", "127.0.0.1:9043"]

In [5]:
from cassandra.cluster import Cluster
from cassandra.connection import DefaultEndPoint
from cassandra.auth import PlainTextAuthProvider
from cassandra.query import dict_factory

cassandra_addresses = [
    h.split(":")[0] for h in cassandra_endpoints
]
cassandra_ports = [
    int(h.split(":")[1]) for h in cassandra_endpoints
]
cassandra_nodes = [
    DefaultEndPoint(addr, port)
    for addr, port in zip(cassandra_addresses, cassandra_ports)
]

auth_provider = PlainTextAuthProvider(
    username=cassandra_user, password=cassandra_password
)
cluster = Cluster(contact_points=cassandra_nodes, auth_provider=auth_provider, control_connection_timeout=60.0)
session = cluster.connect()
session.default_timeout = 60.0
session.row_factory = dict_factory
print(f"✅ Conectado a Cassandra: {cluster.metadata.cluster_name} (Nodos: {len(cluster.metadata.all_hosts())})")

✅ Conectado a Cassandra: cassandra-cluster (Nodos: 3)


In [6]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def cql(line, cell):
    try:
        lines = [l for l in cell.splitlines() if not l.strip().startswith(("#", "//", "--"))]
        cleaned = chr(10).join(lines).strip()
        if "BEGIN BATCH" in cleaned.upper():
            return session.execute(cleaned)
        queries = [q.strip() for q in cleaned.split(";") if q.strip()]
        last_res = None
        for q in queries:
            last_res = session.execute(q)
        return last_res
    except Exception as e:
        print(f"❌ Error ejecutando CQL: {e}")
        raise e

In [7]:
# Limpieza automática de tabla previa para garantizar reproducibilidad
try:
    session.execute("DROP TABLE IF EXISTS blog_red.publicaciones")
    session.execute("DROP KEYSPACE IF EXISTS blog_red")
except Exception as e:
    pass

## Pregunta 1 (DDL) - Creación del Keyspace

Crea un keyspace llamado ```blog_red``` con estrategia de replicación `SimpleStrategy` y factor de replicación `3`.

In [8]:
%%cql
CREATE KEYSPACE IF NOT EXISTS blog_red 
WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 3};

Valida la creación del keyspace.

In [9]:
print("Validando Keyspace...")
session.execute("DESCRIBE KEYSPACE blog_red").one()

Validando Keyspace...


{'keyspace_name': 'blog_red',
 'type': 'keyspace',
 'name': 'blog_red',
 'create_statement': "CREATE KEYSPACE blog_red WITH replication = {'class': 'SimpleStrategy', 'replication_factor': '3'}  AND durable_writes = true;"}

## Pregunta 2 (DDL) - Creación de la Tabla

Crea la tabla ```blog_red.publicaciones``` con la siguiente estructura:
- Clave de partición: `((usuario_id))` (tipo `uuid`)
- Claves de clustering: `fecha_publicacion` (`timestamp`) y `post_id` (`uuid`)
- Orden de clustering: `fecha_publicacion DESC, post_id ASC`
- Columnas adicionales:
  - `titulo` (`text`)
  - `contenido` (`text`)
  - `vistas` (`int`)
  - `es_publico` (`boolean`)
  - `tags` (`set<text>`)
  - `comentarios` (`list<text>`)
  - `metadatos` (`map<text, text>`)

In [10]:
%%cql
CREATE TABLE IF NOT EXISTS blog_red.publicaciones (
    usuario_id uuid,
    fecha_publicacion timestamp,
    post_id uuid,
    titulo text,
    contenido text,
    vistas int,
    es_publico boolean,
    tags set<text>,
    comentarios list<text>,
    metadatos map<text, text>,
    PRIMARY KEY ((usuario_id), fecha_publicacion, post_id) 
) WITH CLUSTERING ORDER BY (fecha_publicacion DESC, post_id ASC);

Valida la creación de la tabla y su orden de clustering.

In [11]:
print("Validando Tabla...")
list(session.execute("""
    SELECT column_name, kind, position, clustering_order
    FROM system_schema.columns
    WHERE keyspace_name='blog_red' AND table_name='publicaciones'
"""))

Validando Tabla...


[{'column_name': 'comentarios',
  'kind': 'regular',
  'position': -1,
  'clustering_order': 'none'},
 {'column_name': 'contenido',
  'kind': 'regular',
  'position': -1,
  'clustering_order': 'none'},
 {'column_name': 'es_publico',
  'kind': 'regular',
  'position': -1,
  'clustering_order': 'none'},
 {'column_name': 'fecha_publicacion',
  'kind': 'clustering',
  'position': 0,
  'clustering_order': 'desc'},
 {'column_name': 'metadatos',
  'kind': 'regular',
  'position': -1,
  'clustering_order': 'none'},
 {'column_name': 'post_id',
  'kind': 'clustering',
  'position': 1,
  'clustering_order': 'asc'},
 {'column_name': 'tags',
  'kind': 'regular',
  'position': -1,
  'clustering_order': 'none'},
 {'column_name': 'titulo',
  'kind': 'regular',
  'position': -1,
  'clustering_order': 'none'},
 {'column_name': 'usuario_id',
  'kind': 'partition_key',
  'position': 0,
  'clustering_order': 'none'},
 {'column_name': 'vistas',
  'kind': 'regular',
  'position': -1,
  'clustering_order': 'n

## Inicialización y Siembra Masiva de Datos Históricos (Faker)

Una vez creada la estructura de la tabla, ejecutamos la siembra masiva de **500 registros históricos** generados sintéticamente con `Faker('es_MX')` para simular la operación y telemetría histórica del sistema en producción. Esta celda está bloqueada y poblará la base de datos para los análisis posteriores.

In [12]:
import datetime, random, uuid
from faker import Faker

fake = Faker("es_MX")
Faker.seed(42)
random.seed(42)

insert_stmt = session.prepare(f"""
INSERT INTO blog_red.publicaciones (usuario_id, fecha_publicacion, post_id, titulo, contenido, vistas, es_publico, tags, comentarios, metadatos)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

# Sembrar 500 registros históricos distribuidos en 10 particiones
preseed_pks = [uuid.UUID(f"f{i:07d}-9999-9999-9999-999999999999") for i in range(1, 11)]
start_date = datetime.datetime(2026, 1, 1, 0, 0, 0)

for p_idx, p_val in enumerate(preseed_pks):
    for i in range(50):
        c1_val = start_date + datetime.timedelta(days=i, hours=(i*3)%24, minutes=(i*17)%60)
        c2_val = uuid.UUID(f"a{p_idx:03d}{i:04d}-8888-8888-8888-888888888888")
        t1_text = fake.sentence(nb_words=4)
        t2_text = fake.paragraph(nb_sentences=2)
        ci_val = 10 + (i * 7) % 900
        cb_val = (i % 2 == 0)
        cset_val = {"historico", "produccion", fake.word()}
        clist_val = ["ingesta", "procesado", "archivado"]
        cmap_val = {"region": fake.state(), "fuente": "cluster_iot"}
        
        session.execute(insert_stmt, (
            p_val, c1_val, c2_val, t1_text, t2_text,
            ci_val, cb_val, cset_val, clist_val, cmap_val
        ))

print(f"✅ Se han sembrado exitosamente 500 registros históricos en blog_red.publicaciones con Faker.")

✅ Se han sembrado exitosamente 500 registros históricos en blog_red.publicaciones con Faker.


## Pregunta 3 (DML) - Inserción Escalar

Inserta el primer registro en ```blog_red.publicaciones```:
- `usuario_id`: ```a1111111-1111-1111-1111-111111111111```
- `fecha_publicacion`: `'2026-03-01 10:00:00+0000'`
- `post_id`: ```c3333333-3333-3333-3333-333333333333```
- `titulo`: `'Introduccion a NoSQL Distribuido'`
- `contenido`: `'Registro inicial del sistema'`
- `vistas`: `150`
- `es_publico`: `true`

In [13]:
%%cql
INSERT INTO blog_red.publicaciones (usuario_id, fecha_publicacion, post_id, titulo, contenido, vistas, es_publico)
VALUES (a1111111-1111-1111-1111-111111111111, 
        '2026-03-01 10:00:00+0000', 
        c3333333-3333-3333-3333-333333333333, 
        'Introduccion a NoSQL Distribuido',
        'Registro inicial del sistema',
        150, 
        true); 

Valida la inserción del primer registro.

In [14]:
print("Validando Registro 1...")
list(session.execute("""SELECT * FROM blog_red.publicaciones
WHERE usuario_id = a1111111-1111-1111-1111-111111111111;"""))

Validando Registro 1...


[{'usuario_id': UUID('a1111111-1111-1111-1111-111111111111'),
  'fecha_publicacion': datetime.datetime(2026, 3, 1, 10, 0),
  'post_id': UUID('c3333333-3333-3333-3333-333333333333'),
  'comentarios': None,
  'contenido': 'Registro inicial del sistema',
  'es_publico': True,
  'metadatos': None,
  'tags': None,
  'titulo': 'Introduccion a NoSQL Distribuido',
  'vistas': 150}]

## Pregunta 4 (DML) - Inserción Compleja con Colecciones

Inserta el segundo registro en ```blog_red.publicaciones``` con colecciones:
- `usuario_id`: ```a1111111-1111-1111-1111-111111111111```
- `fecha_publicacion`: `'2026-03-02 12:00:00+0000'`
- `post_id`: ```d4444444-4444-4444-4444-444444444444```
- `titulo`: `'Modelado Orientado a Consultas'`
- `contenido`: `'Segundo evento registrado con colecciones'`
- `vistas`: `280`
- `es_publico`: `true`
- `tags`: `{'critico', 'auditoria'}`
- `comentarios`: `['inicio', 'verificacion', 'completado']`
- `metadatos`: `{'ambiente': 'produccion', 'region': 'mx-central'}`

In [15]:
%%cql
INSERT INTO blog_red.publicaciones (usuario_id, fecha_publicacion, post_id, titulo, contenido, vistas, es_publico, tags, comentarios, metadatos)
VALUES (a1111111-1111-1111-1111-111111111111,
        '2026-03-02 12:00:00+0000', 
        d4444444-4444-4444-4444-444444444444, 
        'Modelado Orientado a Consultas',
        'Segundo evento registrado con colecciones',
        280, 
        true,
        {'critico', 'auditoria'},
        ['inicio', 'verificacion', 'completado'],
        {'ambiente':'produccion', 'region': 'mx-central'}
);

Valida la inserción con colecciones.

In [16]:
print("Validando Registro 2...")
for r in session.execute("""
    SELECT fecha_publicacion, post_id, titulo
    FROM blog_red.publicaciones
    WHERE usuario_id = a1111111-1111-1111-1111-111111111111
"""):
    print(r)

Validando Registro 2...
{'fecha_publicacion': datetime.datetime(2026, 3, 2, 12, 0), 'post_id': UUID('d4444444-4444-4444-4444-444444444444'), 'titulo': 'Modelado Orientado a Consultas'}
{'fecha_publicacion': datetime.datetime(2026, 3, 1, 10, 0), 'post_id': UUID('c3333333-3333-3333-3333-333333333333'), 'titulo': 'Introduccion a NoSQL Distribuido'}


## Pregunta 5 (DML) - Inserción Temporal con TTL

Inserta el tercer registro en ```blog_red.publicaciones``` para la partición ```b2222222-2222-2222-2222-222222222222``` con expiración temporal:
- `usuario_id`: ```b2222222-2222-2222-2222-222222222222```
- `fecha_publicacion`: `'2026-03-03 15:30:00+0000'`
- `post_id`: ```e5555555-5555-5555-5555-555555555555```
- `titulo`: `'Patrones de Particionamiento'`
- `contenido`: `'Evento volatil temporal'`
- `vistas`: `420`
- `es_publico`: `false`
- `USING TTL 86400`

In [17]:
%%cql
INSERT INTO blog_red.publicaciones (usuario_id, fecha_publicacion, post_id, titulo, contenido, vistas, es_publico)
VALUES (b2222222-2222-2222-2222-222222222222,
        '2026-03-03 15:30:00+0000', 
        e5555555-5555-5555-5555-555555555555, 
        'Patrones de Particionamiento',
        'Evento volatil temporal',
        420, 
        false
)
USING TTL 86400;

Valida la inserción con TTL.

In [18]:
print("Validando Registro con TTL...")
list(session.execute("""
    SELECT titulo, TTL(titulo) FROM blog_red.publicaciones
    WHERE usuario_id = b2222222-2222-2222-2222-222222222222
"""))

Validando Registro con TTL...


[{'titulo': 'Patrones de Particionamiento', 'ttl(titulo)': 86400}]

## Pregunta 6 (DQL) - Consulta Exacta por Partición

Realiza una consulta para obtener todos los registros de la partición ```usuario_id = a1111111-1111-1111-1111-111111111111```.
Guarda el resultado en la variable ```registros_particion_p1``` como una lista de diccionarios.

In [19]:
registros_particion_p1 = list(session.execute('''
SELECT * FROM blog_red.publicaciones
WHERE usuario_id = a1111111-1111-1111-1111-111111111111
'''))

Valida la consulta por partición.

In [20]:
print("Validando consulta DQL P1...")
for r in registros_particion_p1:
    print(r['post_id'],r['fecha_publicacion'], r['titulo'])

Validando consulta DQL P1...
d4444444-4444-4444-4444-444444444444 2026-03-02 12:00:00 Modelado Orientado a Consultas
c3333333-3333-3333-3333-333333333333 2026-03-01 10:00:00 Introduccion a NoSQL Distribuido


## Pregunta 7 (DQL) - Consulta con Rango en Clave de Clustering

Realiza una consulta sobre la partición ```usuario_id = a1111111-1111-1111-1111-111111111111``` filtrando por fecha en la clave de clustering:
`fecha_publicacion >= '2026-03-02 00:00:00+0000'` y `fecha_publicacion <= '2026-03-02 23:59:59+0000'`.
Guarda los resultados en ```registros_rango_fecha```.

In [21]:
registros_rango_fecha = list(session.execute('''
SELECT * FROM blog_red.publicaciones
WHERE usuario_id = a1111111-1111-1111-1111-111111111111
AND fecha_publicacion >= '2026-03-02 00:00:00+0000'
AND fecha_publicacion <= '2026-03-02 23:59:59+0000'
'''))

Valida la consulta de rango de clustering.

In [22]:
print("Validando rango de clustering...")
for r in registros_rango_fecha:
    print(r['post_id'],r['fecha_publicacion'], r['titulo'])

Validando rango de clustering...
d4444444-4444-4444-4444-444444444444 2026-03-02 12:00:00 Modelado Orientado a Consultas


## Pregunta 8 (DQL) - Consulta Multi-Partición con IN

Realiza una consulta para obtener los registros donde ```usuario_id IN (a1111111-1111-1111-1111-111111111111, b2222222-2222-2222-2222-222222222222)```.
Guarda los resultados en ```registros_in_particiones```.

In [23]:
registros_in_particiones = list(session.execute('''
SELECT * FROM blog_red.publicaciones
WHERE usuario_id IN (a1111111-1111-1111-1111-111111111111, b2222222-2222-2222-2222-222222222222)
'''))

Valida la consulta multi-partición IN.

In [24]:
print("Validando consulta IN...")
for r in registros_in_particiones:
    print(r['usuario_id'], r['post_id'], r['fecha_publicacion'], r['titulo'])

Validando consulta IN...
a1111111-1111-1111-1111-111111111111 d4444444-4444-4444-4444-444444444444 2026-03-02 12:00:00 Modelado Orientado a Consultas
a1111111-1111-1111-1111-111111111111 c3333333-3333-3333-3333-333333333333 2026-03-01 10:00:00 Introduccion a NoSQL Distribuido
b2222222-2222-2222-2222-222222222222 e5555555-5555-5555-5555-555555555555 2026-03-03 15:30:00 Patrones de Particionamiento


## Pregunta 9 (DML) - Actualización de Campo Escalar

Modifica el registro de ```a1111111-1111-1111-1111-111111111111``` en fecha `'2026-03-01 10:00:00+0000'` e `post_id = c3333333-3333-3333-3333-333333333333` para actualizar:
`vistas = 999` y `contenido = 'Texto modificado exitosamente'`.

In [25]:
%%cql
UPDATE blog_red.publicaciones SET vistas = 999, contenido = 'Texto modificado exitosamente'
WHERE usuario_id = a1111111-1111-1111-1111-111111111111
AND fecha_publicacion = '2026-03-01 10:00:00+0000'
AND post_id = c3333333-3333-3333-3333-333333333333;

Valida la actualización escalar.

In [26]:
print("Validando UPDATE escalar...")
list(session.execute("""
SELECT vistas, contenido FROM blog_red.publicaciones
WHERE usuario_id = a1111111-1111-1111-1111-111111111111
AND fecha_publicacion = '2026-03-01 10:00:00+0000'
AND post_id = c3333333-3333-3333-3333-333333333333
"""))

Validando UPDATE escalar...


[{'vistas': 999, 'contenido': 'Texto modificado exitosamente'}]

## Pregunta 10 (DML) - Mutación de Colección SET

Agrega el tag `'supervisado'` a la colección ```tags``` del registro ```a1111111-1111-1111-1111-111111111111``` (`'2026-03-02 12:00:00+0000'`, `d4444444-4444-4444-4444-444444444444`).

In [27]:
%%cql
UPDATE blog_red.publicaciones SET tags = tags + {'supervisado'}
WHERE usuario_id = a1111111-1111-1111-1111-111111111111
AND fecha_publicacion = '2026-03-02 12:00:00+0000'
AND post_id = d4444444-4444-4444-4444-444444444444;

Valida la mutación en la colección SET.

In [28]:
print("Validando SET mutation...")
list(session.execute("""
SELECT tags FROM blog_red.publicaciones
WHERE usuario_id = a1111111-1111-1111-1111-111111111111
AND fecha_publicacion = '2026-03-02 12:00:00+0000'
AND post_id = d4444444-4444-4444-4444-444444444444
"""))

Validando SET mutation...


[{'tags': SortedSet(['auditoria', 'critico', 'supervisado'])}]

## Pregunta 11 (DML) - Mutación de Colección MAP

Agrega o actualiza la clave `'estado_revision'` con valor `'aprobado'` en el mapa ```metadatos``` del registro ```a1111111-1111-1111-1111-111111111111``` (`'2026-03-02 12:00:00+0000'`, `d4444444-4444-4444-4444-444444444444`).

In [29]:
%%cql
UPDATE blog_red.publicaciones SET metadatos = metadatos + {'estado_revision': 'aprobado'}
WHERE usuario_id = a1111111-1111-1111-1111-111111111111
AND fecha_publicacion = '2026-03-02 12:00:00+0000'
AND post_id = d4444444-4444-4444-4444-444444444444;

Valida la mutación en la colección MAP.

In [30]:
print("Validando MAP mutation...")
list(session.execute("""
SELECT metadatos FROM blog_red.publicaciones
WHERE usuario_id = a1111111-1111-1111-1111-111111111111
AND fecha_publicacion = '2026-03-02 12:00:00+0000'
AND post_id = d4444444-4444-4444-4444-444444444444
"""))

Validando MAP mutation...


[{'metadatos': OrderedMapSerializedKey([('ambiente', 'produccion'), ('estado_revision', 'aprobado'), ('region', 'mx-central')])}]

## Pregunta 12 (DML) - Eliminación de Columna Individual (Celda)

Elimina únicamente la columna ```contenido``` del registro ```a1111111-1111-1111-1111-111111111111``` en fecha `'2026-03-01 10:00:00+0000'` e `post_id = c3333333-3333-3333-3333-333333333333`.

In [31]:
%%cql
DELETE contenido FROM blog_red.publicaciones 
WHERE usuario_id = a1111111-1111-1111-1111-111111111111
AND fecha_publicacion = '2026-03-01 10:00:00+0000'
AND post_id = c3333333-3333-3333-3333-333333333333;

Valida la eliminación de la celda.

In [32]:
print("Validando DELETE columna...")
list(session.execute("""
SELECT contenido FROM blog_red.publicaciones
WHERE usuario_id = a1111111-1111-1111-1111-111111111111
AND fecha_publicacion = '2026-03-01 10:00:00+0000'
AND post_id = c3333333-3333-3333-3333-333333333333
"""))

Validando DELETE columna...


[{'contenido': None}]

## Pregunta 13 (DML) - Eliminación de Clave en Colección MAP

Elimina la clave `'region'` de la colección ```metadatos``` del registro ```a1111111-1111-1111-1111-111111111111``` (`'2026-03-02 12:00:00+0000'`, `d4444444-4444-4444-4444-444444444444`).

In [33]:
%%cql
DELETE metadatos['region'] FROM blog_red.publicaciones
WHERE usuario_id = a1111111-1111-1111-1111-111111111111
 AND fecha_publicacion = '2026-03-02 12:00:00+0000'
 AND post_id = d4444444-4444-4444-4444-444444444444;

Valida la eliminación de la clave del mapa.

In [34]:
print("Validando DELETE map key...")
list(session.execute("""
SELECT metadatos FROM blog_red.publicaciones
WHERE usuario_id = a1111111-1111-1111-1111-111111111111
AND fecha_publicacion = '2026-03-02 12:00:00+0000'
AND post_id = d4444444-4444-4444-4444-444444444444
"""))

Validando DELETE map key...


[{'metadatos': OrderedMapSerializedKey([('ambiente', 'produccion'), ('estado_revision', 'aprobado')])}]

## Pregunta 14 (DML) - Eliminación de Fila Completa

Elimina la fila completa correspondiente a ```a1111111-1111-1111-1111-111111111111``` en fecha `'2026-03-01 10:00:00+0000'` e `post_id = c3333333-3333-3333-3333-333333333333`.

In [35]:
%%cql
DELETE FROM blog_red.publicaciones 
WHERE usuario_id = a1111111-1111-1111-1111-111111111111
AND fecha_publicacion = '2026-03-01 10:00:00+0000'
AND post_id = c3333333-3333-3333-3333-333333333333

Valida la eliminación de la fila.

In [36]:
print("Validando DELETE fila...")
list(session.execute("""
SELECT * FROM blog_red.publicaciones
WHERE usuario_id = a1111111-1111-1111-1111-111111111111
"""))

Validando DELETE fila...


[{'usuario_id': UUID('a1111111-1111-1111-1111-111111111111'),
  'fecha_publicacion': datetime.datetime(2026, 3, 2, 12, 0),
  'post_id': UUID('d4444444-4444-4444-4444-444444444444'),
  'comentarios': ['inicio', 'verificacion', 'completado'],
  'contenido': 'Segundo evento registrado con colecciones',
  'es_publico': True,
  'metadatos': OrderedMapSerializedKey([('ambiente', 'produccion'), ('estado_revision', 'aprobado')]),
  'tags': SortedSet(['auditoria', 'critico', 'supervisado']),
  'titulo': 'Modelado Orientado a Consultas',
  'vistas': 280}]

## Pregunta 15 (DML) - Transacción Atómica con BATCH

Ejecuta un bloque ```BEGIN BATCH ... APPLY BATCH``` que realice de forma atómica:
1. La inserción de un nuevo registro para ```b2222222-2222-2222-2222-222222222222``` con `fecha_publicacion = '2026-03-04 18:00:00+0000'`, `post_id = c3333333-3333-3333-3333-333333333333`, `titulo = 'Batch Event'`, `contenido = 'Batch Detalle'`, `vistas = 777`, `es_publico = true`.
2. La actualización de `vistas = 888` en el registro de ```b2222222-2222-2222-2222-222222222222``` (`'2026-03-03 15:30:00+0000'`, `e5555555-5555-5555-5555-555555555555`).

In [37]:
%%cql
BEGIN BATCH
INSERT INTO blog_red.publicaciones (usuario_id, fecha_publicacion, post_id, titulo, contenido, vistas, es_publico)
VALUES (b2222222-2222-2222-2222-222222222222,
        '2026-03-04 18:00:00+0000',
        c3333333-3333-3333-3333-333333333333, 
        'Batch Event',
        'Batch Detalle',
        777,
        true);
UPDATE blog_red.publicaciones SET vistas = 888
WHERE usuario_id = b2222222-2222-2222-2222-222222222222
AND fecha_publicacion = '2026-03-03 15:30:00+0000'
AND post_id = e5555555-5555-5555-5555-555555555555;
APPLY BATCH;

Valida la ejecución del BATCH atómico.

In [38]:
print("Validando BATCH atómico...")
for r in session.execute("""
    SELECT fecha_publicacion, post_id, titulo, vistas
    FROM blog_red.publicaciones
    WHERE usuario_id = b2222222-2222-2222-2222-222222222222
"""):
    print(r)

Validando BATCH atómico...
{'fecha_publicacion': datetime.datetime(2026, 3, 4, 18, 0), 'post_id': UUID('c3333333-3333-3333-3333-333333333333'), 'titulo': 'Batch Event', 'vistas': 777}
{'fecha_publicacion': datetime.datetime(2026, 3, 3, 15, 30), 'post_id': UUID('e5555555-5555-5555-5555-555555555555'), 'titulo': 'Patrones de Particionamiento', 'vistas': 888}
